In [1]:
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from typing import Optional
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import (ChatPromptTemplate,
                                    PromptTemplate,
                                    SystemMessagePromptTemplate,
                                    HumanMessagePromptTemplate)

In [2]:
load_dotenv()

llm =ChatOllama(
    base_url="http://localhost:11434",
    model = "qwen3"
)

In [3]:
class Joke(BaseModel):
    """Joke to tell user"""
    setup: str = Field(description="The setup of the joke")
    punchline: str = Field(description="The punchline of the joke")
    rating: Optional[int] = Field(description="The rating of the joke from 1 to 10", default=None)


In [4]:
parser = PydanticOutputParser(pydantic_object=Joke)
parser.get_format_instructions()

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"description": "Joke to tell user", "properties": {"setup": {"description": "The setup of the joke", "title": "Setup", "type": "string"}, "punchline": {"description": "The punchline of the joke", "title": "Punchline", "type": "string"}, "rating": {"anyOf": [{"type": "integer"}, {"type": "null"}], "default": null, "description": "The rating of the joke from 1 to 10", "title": "Rating"}}, "required": ["setup", "punchline"]}\n```'

In [5]:
prompt = PromptTemplate(
    template = '''
    Answer the user query with a joke. Here is your formatting instructions: {format_instructions}
    User Query: {query}

    Answer:
    ''',
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)
chain = prompt | llm |parser
output = chain.invoke({"query": "Tell me a joke about programming"})

In [6]:
print(output)

setup='Why do programmers always mix up Halloween and Christmas?' punchline='Because Oct 31 == Dec 25!' rating=9


In [7]:
from langchain_core.output_parsers import JsonOutputParser
class Movie(BaseModel):
    """Details about a movie"""
    title: str = Field(description = "The title of the movie")
    director: str = Field(description = "The director of the movie")
    year: int = Field(description = "The year the movie was released")
    genre: str = Field(description = "The genre of the movie")

movie_parser = JsonOutputParser(pydantic_object=Movie)
movie_prompt = PromptTemplate(
    template = '''
    Answer the user query with details about a movie. Here is your formatting instructions: {format_instructions}
    User Query: {query} 
    Answer: 
    ''',
    input_variables=["query"],
    partial_variables={"format_instructions": movie_parser.get_format_instructions()}
)
movie_chain = movie_prompt | llm | movie_parser
movie_output = movie_chain.invoke({"query": "Tell me about the movie Inception"})
print(movie_output)

{'title': 'Inception', 'director': 'Christopher Nolan', 'year': 2010, 'genre': 'Science Fiction'}


In [8]:
from langchain_core.prompts import ChatPromptTemplate

# Reuse the existing parser and Joke model
parser = PydanticOutputParser(pydantic_object=Joke)

# ChatPromptTemplate with system + human messages
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a comedian. Answer only with a joke in the requested format.\n{format_instructions}"),
    ("human", "{query}")
]).partial(format_instructions=parser.get_format_instructions())

chain = chat_prompt | llm | parser

output = chain.invoke({"query": "Tell me a joke about Python"})
print(output)
# Joke(setup='...', punchline='...', rating=8)


setup='Why do Python programmers always seem calm?' punchline="Because they're always in a 'for' loop, taking it easy." rating=7
